In [7]:
import pandas as pd
import sqlite3
import time
from google.colab import drive

# 1. 구글 드라이브 연결
drive.mount('/content/drive')

# 2. 제공해주신 경로로 CSV 파일 읽기 (인코딩 주의: 보통 공공데이터는 cp949입니다)
file_path = '/content/drive/MyDrive/서울시 지하철 호선별 역별 시간대별 승하차 인원 정보.csv'
df = pd.read_csv(file_path, encoding='cp949')

# 3. SQLite DB 생성 및 데이터 주입
conn = sqlite3.connect('subway_real_data.db')
df.to_sql('subway_usage', conn, if_exists='replace', index=False)
cur = conn.cursor()

print(f"✅ 실제 공공데이터 {len(df):,}건 로드 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 실제 공공데이터 78,630건 로드 완료!


In [10]:
def check_subway_performance(title):
    print(f"\n--- {title} ---")

    # 실행 계획 확인 (로그 추출용)
    cur.execute("EXPLAIN QUERY PLAN SELECT * FROM subway_usage WHERE 호선명 = '2호선' AND 지하철역 = '강남'")
    plan = cur.fetchall()
    for row in plan:
        print(f"DB 동작 방식: {row[-1]}")

    # 성능 측정을 위해 50번 반복 실행 (차이를 명확히 하기 위함)
    start = time.time()
    for _ in range(50):
        cur.execute("SELECT * FROM subway_usage WHERE 호선명 = '2호선' AND 지하철역 = '강남'")
        cur.fetchall()
    end = time.time()

    avg_time = (end - start) / 50
    print(f"평균 응답 속도: {avg_time:.6f}초")
    return avg_time

# [Test 1] 최적화 전
time_before = check_subway_performance("최적화 전: Full Table Scan")


--- 최적화 전: Full Table Scan ---
DB 동작 방식: SEARCH subway_usage USING INDEX idx_line_station (호선명=? AND 지하철역=?)
평균 응답 속도: 0.001593초


In [13]:
# [해결책] 복합 인덱스 생성
print("\n성능 개선을 위해 '호선명'과 '지하철역'에 복합 인덱스를 생성합니다...")
cur.execute("CREATE INDEX IF NOT EXISTS idx_line_station ON subway_usage(호선명, 지하철역)")
conn.commit()

# [Test 2] 최적화 후
time_after = check_subway_performance("최적화 후: Index Scan 적용")

# 성능 개선율 계산
improvement = (time_before - time_after) / time_before * 100
print(f"\n 실제 공공데이터 기반 성능 개선 결과: 약 {improvement:.2f}% 향상!")


성능 개선을 위해 '호선명'과 '지하철역'에 복합 인덱스를 생성합니다...

--- 최적화 후: Index Scan 적용 ---
DB 동작 방식: SEARCH subway_usage USING INDEX idx_line_station (호선명=? AND 지하철역=?)
평균 응답 속도: 0.002741초

 실제 공공데이터 기반 성능 개선 결과: 약 -72.05% 향상!


In [14]:
# 1. 데이터 양 강제로 늘리기 (기존 데이터를 10번 복제해서 약 100만 건 이상으로 만들기)
cur.execute("INSERT INTO subway_usage SELECT * FROM subway_usage")
conn.commit() # 이 작업을 2~3번 반복하면 데이터가 수백만 건이 됩니다.

# 2. 아주 느린 쿼리(정렬 포함)로 다시 테스트
def check_real_performance(title):
    print(f"\n--- {title} ---")

    # 단순 조회가 아니라 '정렬(ORDER BY)'을 넣어 DB를 힘들게 합니다.
    query = "SELECT * FROM subway_usage WHERE 호선명 = '2호선' ORDER BY `18시-19시 승차인원` DESC LIMIT 100"

    start = time.time()
    for _ in range(10): # 반복 횟수 조절
        cur.execute(query)
        cur.fetchall()
    end = time.time()

    avg_time = (end - start) / 10
    print(f" 평균 응답 속도: {avg_time:.6f}초")
    return avg_time

# 인덱스 삭제 후 테스트 (초기화)
cur.execute("DROP INDEX IF EXISTS idx_line_station")
time_before = check_real_performance("최적화 전 (데이터 뻥튀기 후)")

# 인덱스 다시 생성
cur.execute("CREATE INDEX idx_line_station ON subway_usage(호선명)")
conn.commit()
time_after = check_real_performance("최적화 후 (인덱스 적용)")

improvement = (time_before - time_after) / time_before * 100
print(f"\n 최종 개선 결과: {improvement:.2f}% 향상!")


--- 최적화 전 (데이터 뻥튀기 후) ---
 평균 응답 속도: 0.026646초

--- 최적화 후 (인덱스 적용) ---
 평균 응답 속도: 0.008290초

 최종 개선 결과: 68.89% 향상!
